In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ACQ_H.xpt"
csv_file = DATA_DIR / "ACQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 ACQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ACD011A",
    "ACD011B",
    "ACD011C",
    "ACD040",
    "ACD110"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ACQ_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9230:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,230 rows x 6 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# Attached ACQ_H contains NO genuine decimal variables.
#
# All six fields are identifiers / categorical codes.
#
# Therefore all six can safely become nullable Int64.
#
# IMPORTANT:
#
# This does NOT change the general rule for other NHANES
# datasets:
#
#     Dietary decimals
#     Lab decimals
#     Examination measurements
#     Sample weights
#
# must remain genuine decimal fields where applicable.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(DECIMAL_COLUMNS) != 0:

    raise ValueError(
        "ACQ_H should contain no decimal variables."
    )


if len(INTEGER_COLUMNS) != 6:

    raise ValueError(
        "Expected exactly six whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached ACQ_H contains NONE.
#
# Therefore NO tiny -> zero repair is required.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all ACQ_H fields to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Tiny XPORT artifacts:",
    tiny_total
)


print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in ACQ_H."
    )


if true_zero_total > 0:

    print(
        "\nOrdinary zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected ordinary zeros "
        "found in ACQ_H."
    )


print(
    "PASS: No tiny artifacts and no ordinary zeros."
)


# ============================================================
# 6. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# This check MUST happen before converting to Int64.
#
# np.round() is used ONLY for detecting fractions.
#
# It does NOT modify the data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in ACQ_H. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 8. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All six ACQ_H variables are proven whole-number fields.
#
# NO .round() is used for conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 6 ACQ_H fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 9. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        "Observed:",
        actual_values
    )


    print(
        "Missing:",
        actual_missing
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 10. CDC CHECK - ACD011A
# ============================================================

#
# Speak English at home.
#
# Applies to:
#
# Non-Hispanic White
# Non-Hispanic Black
# Other race / multiracial
#
# CDC:
#
# 1 = English = 5,759
# Missing     = 3,471
#

check_frequency(
    "ACD011A",
    {
        1: 5759
    },
    3471
)


# ============================================================
# 11. CDC CHECK - ACD011B
# ============================================================

#
# Speak Spanish at home.
#
# "Check all that apply" variable.
#
# CDC:
#
# 8 = Spanish = 16
# Missing     = 9,214
#

check_frequency(
    "ACD011B",
    {
        8: 16
    },
    9214
)


# ============================================================
# 12. CDC CHECK - ACD011C
# ============================================================

#
# Speak other language at home.
#
# CDC:
#
# 9 = Other = 171
# Missing   = 9,059
#

check_frequency(
    "ACD011C",
    {
        9: 171
    },
    9059
)


# ============================================================
# 13. CDC CHECK - ACD040
# ============================================================

#
# Language spoken at home - Hispanic participants.
#
# CDC:
#
# 1 = Only Spanish                 548
# 2 = More Spanish than English   321
# 3 = Both equally                444
# 4 = More English than Spanish   483
# 5 = Only English                573
# 7 = Refused                       1
# 9 = Don't know                    4
#
# Missing = 6,856
#

EXPECTED_ACD040 = {
    1: 548,
    2: 321,
    3: 444,
    4: 483,
    5: 573,
    7: 1,
    9: 4
}


check_frequency(
    "ACD040",
    EXPECTED_ACD040,
    6856
)


# ============================================================
# 14. CDC CHECK - ACD110
# ============================================================

#
# Language spoken at home -
# single-race Non-Hispanic Asian participants.
#
# CDC:
#
# 1 = Only non-English language       370
# 2 = More non-English than English    91
# 3 = Both equally                     93
# 4 = More English than non-English   119
# 5 = Only English                    334
#
# Missing = 8,223
#

EXPECTED_ACD110 = {
    1: 370,
    2: 91,
    3: 93,
    4: 119,
    5: 334
}


check_frequency(
    "ACD110",
    EXPECTED_ACD110,
    8223
)


# ============================================================
# 15. VALIDATE ALLOWED VALUES
# ============================================================

#
# Protect against unexpected codes.
#

ALLOWED_VALUES = {

    "ACD011A": {
        1
    },

    "ACD011B": {
        8
    },

    "ACD011C": {
        9
    },

    "ACD040": {
        1,
        2,
        3,
        4,
        5,
        7,
        9
    },

    "ACD110": {
        1,
        2,
        3,
        4,
        5
    }
}


print("\n--- ALLOWED VALUE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


    print(
        f"PASS: {col} contains only expected codes."
    )


# ============================================================
# 16. CONFIRM NO ZERO VALUES CREATED
# ============================================================

#
# Original XPT:
#
#     tiny values = 0
#     ordinary zeros = 0
#
# Therefore the conversion must not create any zero values.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO CHECK ---")


print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zeros were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 17. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ACQ_H requires NO numerical data correction.
#
# Therefore every populated value must remain
# numerically identical to the source XPT.
#
# Int64 conversion changes only CSV presentation:
#
#     73557.0 -> 73557
#     1.0     -> 1
#     8.0     -> 8
#
# It does not change the numeric meaning.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 6 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 18. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity rule:
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# ACQ_H itself contains no decimals, but this same
# export pattern protects consistency with Dietary,
# Laboratory and Examination files that do.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 19. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 6 CSV variables round-trip "
    "to exact source XPT values."
)


# ============================================================
# 21. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 22. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 23. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 24. RAW CSV .0 CHECK
# ============================================================

#
# For ACQ_H specifically, a global ".0" check is safe
# because every variable has been proven whole-number.
#
# Expected examples:
#
#     SEQN
#     73557.0 -> 73557
#
#     ACD011A
#     1.0 -> 1
#
#     ACD011B
#     8.0 -> 8
#
#     ACD011C
#     9.0 -> 9
#
# DO NOT copy this global rule to mixed decimal datasets.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any ACQ_H field."
)


# ============================================================
# 25. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO CHECK ---")


print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 26. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 27. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "ACQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Acculturation"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "ACD011A English:",
    int(
        df["ACD011A"]
        .eq(1)
        .sum()
    )
)


print(
    "ACD011B Spanish:",
    int(
        df["ACD011B"]
        .eq(8)
        .sum()
    )
)


print(
    "ACD011C Other language:",
    int(
        df["ACD011C"]
        .eq(9)
        .sum()
    )
)


print(
    "ACD040 nonmissing:",
    int(
        df["ACD040"]
        .notna()
        .sum()
    )
)


print(
    "ACD110 nonmissing:",
    int(
        df["ACD110"]
        .notna()
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 ACQ_H XPT -> CSV
Rows: 9,230
Columns: 6
PASS: Structure = 9,230 rows x 6 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
Whole-number/code fields: ['SEQN', 'ACD011A', 'ACD011B', 'ACD011C', 'ACD040', 'ACD110']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: No tiny artifacts and no ordinary zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
ACD011A: 0 fractional observations
ACD011B: 0 fractional observations
ACD011C: 0 fractional observations
ACD040: 0 fractional observations
ACD110: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 73557
Maximum: 83731
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83731 validated.

PASS: All 6 ACQ_H fields converted to nullable Int64 without rounding.

--- ACD011A ---
Observed: {np.int64(1): 5759}
Missing: 3471
PASS: ACD011A matches CDC.

--- ACD011B ---
Observed: {np.int64(8): 16}
Missing: 9214
PASS: ACD011B matches CDC.

--- A